In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"

In [2]:
import re
import time
import random
import warnings
from collections import Counter
import numpy as np, pandas as pd, polars as pl

import torch
import vllm
from vllm import LLM, SamplingParams

import kaggle_evaluation.aimo_2_inference_server

INFO 07-27 07:24:10 __init__.py:183] Automatically detected platform cuda.


In [3]:
warnings.simplefilter('ignore')
print('PyTorch version:', torch.__version__)
print('vLLM:', vllm.__version__)

PyTorch version: 2.5.1+cu124
vLLM: 0.7.1


In [4]:
# def seed_everything(seed):
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cudnn.deterministic = True
# seed_everything(seed=0)

# start_time = time.time()
# cutoff_time = start_time + (4 * 60 + 45) * 60
# cutoff_times = [int(x) for x in np.linspace(cutoff_time, start_time + 60 * 60, 50 + 1)]

In [5]:
# llm = LLM(
#     model="/kaggle/input/mistral-small-24b/transformers/mistral-small-24b-base-2501/1",
#      tensor_parallel_size=4,
#     gpu_memory_utilization=0.8,
#     max_model_len=2048,
#     dtype="float16"
# )

# # Step 4: Define generation function
# def generate(prompt, max_tokens=150, temperature=0.7):
#     # Format prompt for chat
#     formatted_prompt = f"<|start_header_id|>user<|end_header_id|>\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
#     # Sampling parameters
#     sampling_params = SamplingParams(
#         temperature=temperature,
#         max_tokens=max_tokens,
#         top_p=0.9,
#         stop=["<|eot_id|>"]
#     )
    
#     # Generate
#     outputs = llm.generate([formatted_prompt], sampling_params)
#     return outputs[0].outputs[0].text.strip()

# # Step 5: Test it!
# response = generate("আপনি কে? বাংলায় উত্তর দিন।")
# print("Response:", response)

# # Quick test examples
# test_prompts = [
#     "বাংলাদেশের রাজধানী কী?",
#     "একটি ছোট কবিতা লিখুন।",
#     "ভালো খাবারের রেসিপি দিন।"
# ]

# for prompt in test_prompts:
#     response = generate(prompt, max_tokens=100)
#     print(f"\nQ: {prompt}")
#     print(f"A: {response}")

# print("\n✅ vLLM setup complete!")

In [6]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True

seed_everything(seed=0)
start_time = time.time()
cutoff_time = start_time + (4 * 60 + 45) * 60
cutoff_times = [int(x) for x in np.linspace(cutoff_time, start_time + 60 * 60, 50 + 1)]

# KEY FIX: Add tensor_parallel_size=4 to distribute across all 4 GPUs
llm = LLM(
    model="/kaggle/input/mistral-small-24b/transformers/mistral-small-24b-base-2501/1",
    tensor_parallel_size=4,  # Distribute across 4 GPUs
    gpu_memory_utilization=0.8,
    max_model_len=4000
)

# Step 4: Define generation function
# def generate(prompt, max_tokens=150, temperature=0.7):
#     # Format prompt for chat
#     formatted_prompt = f"<|start_header_id|>user<|end_header_id|>\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
#     # Sampling parameters
#     sampling_params = SamplingParams(
#         temperature=temperature,
#         max_tokens=max_tokens,
#         top_p=0.9,
#         stop=["<|eot_id|>"]
#     )
    
#     # Generate
#     outputs = llm.generate([formatted_prompt], sampling_params)
#     return outputs[0].outputs[0].text.strip()

# # Step 5: Test it!
# response = generate("আপনি কে? বাংলায় উত্তর দিন।")
# print("Response:", response)

# # Quick test examples
# test_prompts = [
#     "বাংলাদেশের রাজধানী কী?",
#     "একটি ছোট কবিতা লিখুন।",
#     "ভালো খাবারের রেসিপি দিন।"
# ]

# for prompt in test_prompts:
#     response = generate(prompt, max_tokens=100)
#     print(f"\nQ: {prompt}")
#     print(f"A: {response}")

# print("\n✅ vLLM setup complete!")

INFO 07-27 07:24:45 config.py:526] This model supports multiple tasks: {'generate', 'classify', 'embed', 'score', 'reward'}. Defaulting to 'generate'.
INFO 07-27 07:24:45 config.py:1383] Defaulting to use mp for distributed inference
INFO 07-27 07:24:45 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='/kaggle/input/mistral-small-24b/transformers/mistral-small-24b-base-2501/1', speculative_config=None, tokenizer='/kaggle/input/mistral-small-24b/transformers/mistral-small-24b-base-2501/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4000, download_dir=None, load_format=auto, tensor_parallel_size=4, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=Observ

Loading safetensors checkpoint shards:   0% Completed | 0/10 [00:00<?, ?it/s]


(VllmWorkerProcess pid=533) INFO 07-27 07:27:34 model_runner.py:1116] Loading model weights took 11.0233 GB
INFO 07-27 07:27:35 model_runner.py:1116] Loading model weights took 11.0233 GB
(VllmWorkerProcess pid=525) INFO 07-27 07:27:35 model_runner.py:1116] Loading model weights took 11.0233 GB
(VllmWorkerProcess pid=528) INFO 07-27 07:27:35 model_runner.py:1116] Loading model weights took 11.0233 GB
(VllmWorkerProcess pid=533) INFO 07-27 07:27:52 worker.py:266] Memory profiling takes 16.59 seconds
(VllmWorkerProcess pid=528) (VllmWorkerProcess pid=533) INFO 07-27 07:27:52 worker.py:266] Memory profiling takes 16.59 seconds
(VllmWorkerProcess pid=525) INFO 07-27 07:27:52 worker.py:266] the current vLLM instance can use total_gpu_memory (22.28GiB) x gpu_memory_utilization (0.80) = 17.82GiB
(VllmWorkerProcess pid=528) (VllmWorkerProcess pid=533) INFO 07-27 07:27:52 worker.py:266] Memory profiling takes 16.22 seconds
INFO 07-27 07:27:52 worker.py:266] model weights take 11.02GiB; non_torc

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:38<00:00,  1.16s/it]

(VllmWorkerProcess pid=525) 

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:38<00:00,  1.09s/it]

INFO 07-27 07:28:34 model_runner.py:1563] Graph capturing finished in 38 secs, took 0.51 GiB
INFO 07-27 07:28:34 model_runner.py:1563] Graph capturing finished in 38 secs, took 0.51 GiB
(VllmWorkerProcess pid=533) (VllmWorkerProcess pid=528) INFO 07-27 07:28:34 model_runner.py:1563] Graph capturing finished in 38 secs, took 0.51 GiB
INFO 07-27 07:28:34 model_runner.py:1563] Graph capturing finished in 38 secs, took 0.51 GiB
INFO 07-27 07:28:34 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 59.49 seconds


In [ ]:
# Your existing model setup code remains the same...
# (Keep all your seed_everything, LLM initialization, etc.)

# Modified generate function to handle system + user prompts
def generate_response(messages, max_tokens=1500, temperature=0.1):
    """
    Generate response using system and user messages
    messages: List of dicts with 'role' and 'content' keys
    """
    # Extract system and user messages
    system_content = ""
    user_content = ""
    
    for message in messages:
        if message["role"] == "system":
            system_content = message["content"]
        elif message["role"] == "user":
            user_content = message["content"]
    
    # Format the prompt with system and user content
    formatted_prompt = f"""<|start_header_id|>system<|end_header_id|>
{system_content}<|eot_id|><|start_header_id|>user<|end_header_id|>
{user_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    
    # Sampling parameters
    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=0.1,
        stop=["<|eot_id|>"]
    )
    
    # Generate
    outputs = llm.generate([formatted_prompt], sampling_params)
    return outputs[0].outputs[0].text.strip()

# Function to create the Bengali contextual/parametric system
def create_bengali_system_prompt():
    return '''You are tasked with generating both parametric and contextual answers based on a Bengali context. Contextual Answer: Derive strictly from the given context. If the context lacks sufficient info, reply: "Context does not provide enough information." Parametric Answer: Use pre-trained knowledge only; do not refer to the context. If information is missing, make reasonable assumptions and state them. If not possible, reply: "None." Key Note: In the context, a word, year, or number might be incorrect. However, you must extract contextual answers as given in the context, even if it is wrong. On the contrary, you should answer parametric answers correctly while correcting error of context based on your knowledge. Thought Process: Think step by step to ensure clarity. Explain how the contextual and parametric answers were derived. After explaining the derivation process, make sure to write "end of thought process" and then provide your response. Response Format: Contextual Answer: {Answer based only on the context.} Parametric Answer: {Answer based on knowledge without referencing the context.} Reasoning: Explain how both answers were derived step by step. Example: Context: "বাংলাদেশের রাজধানী চট্টগ্রাম।" Question: "বাংলাদেশের রাজধানীর নাম কী?" Output that you will generate: Reasoning: The context explicitly states the capital is Chattogram, so the contextual answer is "চট্টগ্রাম।" Based on my knowledge, the capital is Dhaka, correcting the error in the context. End of thought process Contextual Answer: "চট্টগ্রাম।" Parametric Answer: "ঢাকা।" '''

# Function to process Bengali context-based queries
def process_bengali_query(context, question):
    """
    Process a Bengali query with context to get both contextual and parametric answers
    """
    system_prompt = create_bengali_system_prompt()
    
    # Create user message with context and question
    user_message = f"Context: \"{context}\"\nQuestion: \"{question}\""
    
    # Create messages list
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    # Generate response
    response = generate_response(messages, max_tokens=1500, temperature=0.3)
    return response

# Example usage
if __name__ == "__main__":
    # Test with the provided example
    context = "বাংলাদেশের রাজধানী চট্টগ্রাম।"
    question = "বাংলাদেশের রাজধানীর নাম কী?"
    
    print("Testing Bengali Context-Based Response:")
    print(f"Context: {context}")
    print(f"Question: {question}")
    print("\nResponse:")
    response = process_bengali_query(context, question)
    print(response)
    
    print("\n" + "="*50 + "\n")
    
    # Additional test cases
    test_cases = [
        {
            "context": '''লুবানা বন্দর হল ইসরাইলের তিনটি প্রধান বৃহত্তম আন্তর্জাতিক সমুদ্রবন্দরের মধ্যে একটি, অন্য দুটি হল আশদোদ বন্দর এবং ইয়ালাত বন্দর। এটি একটি প্রাকৃতিক গভীর জলের পোতাশ্রয় যা সারা বছর ধরে পরিচালনা করা হয়, এবং যাত্রী এবং বাণিজ্যিক উভয় জাহাজ পরিসেবা প্রদান করে। এটি মালভূমির পরিমাণ অনুযায়ী পূর্বাংশের ভূমধ্যসাগরীয় বৃহত্তম বন্দর এবং এটি বছরে ২৬ মিলিয়ন টন পণ্যসম্ভার পরিচালনা করে। লুবানার মধ্যে ক্রুজ জাহাজ ডেকে বছরে ৫,০০০ জাহাজের ক্রমবর্ধমান সংখ্যা সহ, বন্দরে ১,০০০ জনের উপর নিযুক্ত।  লুবানা বন্দরটি লুবানা শহরের কেন্দ্রস্থল উপকূলে উত্তরে ভূমধ্যসাগরের উত্তরে অবস্থিত এবং নগরটির কেন্দ্রীয় তীর বরাবর প্রায় ২ কিলোমিটার পর্যন্ত বিস্তৃত, সামরিক, শিল্প ও বাণিজ্যিক অঞ্চল থেকে পরবর্তীকালের-ছোট যাত্রী ক্রুজিং সুবিধা থেকে শুরু করে। লুবানা বন্দরের লুবানা উপসাগর প্রাগৈতিহাসিক সময় থেকে নাবিকদের জন্য একটি আশ্রয় হয়েছে। যখন ১০০ খ্রিষ্টাব্দে ক্রুসেডাররা হাযইফা জয় করে, তখন এটি গালিলের রাজধানী তিবিরিয়া নামে একটি গুরুত্বপূর্ণ শহর ও প্রধান বন্দর হয়ে ওঠে। মমলুকের রাজত্বকালে বন্দর দুর্ভোগে পড়ে এবং ১৮ শতকের একটি পাইরেট খালের খ্যাতি অর্জন করে। লুবানা বন্দর অনেক পণ্যসম্ভার টার্মিনাল আছে, এবং একযোগে অনেক জাহাজ সার্ভিসিং করতে সক্ষম। একটি রেলপথ মালবাহী টার্মিনাল পোর্ট ভিতরে এবং সারা দেশে পণ্য পরিবহনের জন্য ব্যবহার করা হয়। বন্দরটি একটি যাত্রী টার্মিনাল, মাছ ধরার হুইফ, ইয়ট ক্লাব, স্পোর্টস মেরিনা এবং রাসায়নিক টার্মিনালও রয়েছে। ২০১৩ সালে, বন্দর ১.৩৬ মিলিয়ন টিইইউ, এবং সেইসাথে ২৫৩,৫২৪ জন যাত্রী সহ ২৬ মিলিয়ন টন কার্গো পরিবহন করে। বন্দরটি ২০১০ সালে "কারমেল পোর্ট" সম্প্রসারণ প্রোগ্রামে প্রথম পর্যায়ে খোলা হয় যা একটি নতুন পণ্যসম্ভার টার্মিনাল নির্মাণে জড়িত ছিল যার মধ্যে ৯২০০ টিইইউ কনটেইনার জাহাজ (১৫.৫ মিটার ভা ৫১ ফুট) ড্রাফ্ট পরিচালনা করতে সক্ষম একটি ৭০০ মিলিয়ন ঘনফুট) পাশাপাশি একটি দ্বিতীয় ২৫০ মিটার (৮২০ ফুট) ঘূর্ণি প্লাস পাশাপাশি সমর্থন এবং স্টোরেজ এলাকায় খোলার হিসাবে। নতুন সুবিধা ৫০০,০০০ টিইইউ দ্বারা পোর্টের বার্ষিক কন্টেইনার হ্যান্ডলিং ক্ষমতা প্রসারিত করবে। এই নতুন টার্মিনালটি নির্মাণ ১.৮ বিলিয়ন ইসরায়েলি টাকা (প্রায় মার্কিন $ ৫০০ মিলিয়ন) এবং সম্পূর্ণ করতে পাঁচ বছর সময় লাগবে।  বন্দরটি মার্কিন যুক্তরাষ্ট্র ছয়টি ফ্লিট-এর জন্য সুবিধা বজায় রাখে। পোর্ট ক্রুজ এবং ফেরি যাত্রী বহনকারী একটি আধুনিক যাত্রীবাহী টার্ম রয়েছে। টার্মিনাল একটি অপেক্ষা এলাকা, শুল্কমুক্ত দোকান, স্যুভেনির দোকান, ক্যাফেটেরিয়া, ভ্যাট প্রতিযোগিতা কাউন্টার, মুদ্রা বিনিময়, ফ্রি বেতার ইন্টারনেট, পার্কিং, এবং যাত্রীদের জন্য অন্যান্য পরিষেবা প্রদান করে। টার্মিনালটির কাছাকাছি এলাকাটি যাত্রীদের জন্য চমৎকার পাবলিক ট্রানজিট সংযোগ প্রদান করে। লুবানা সেন্ট্রাল রেলওয়ে স্টেশনটি টার্মিনালের সংলগ্ন এবং লুবানা অঞ্চলে এবং এর পরের দিনগুলোতে দিনে ২৪ ঘণ্টা প্রতিদিন ২৪ টি যাত্রী ট্রেন চালায়। অতিরিক্ত পাবলিক ট্রানজিট সংযোগগুলি রেলওয়ে স্টেশন বা হাজ্জম রোডের বাস বা ট্যাক্সিতে পাওয়া যায়, যা হাটহাওয়ারের প্রধান ঘুরে অবস্থিত, যা স্টেশনটির সামনে অবস্থিত। কারমেলিট এর কিকার প্যারিস সাবওয়ে স্টেশন হাঁটার দূরত্ব মধ্যে এবং এছাড়াও কর্মিল পর্বত উপরে সুবিধাজনক প্রবেশাধিকার অনুমতি দেয়।''',
            "question": "ইসরাইলের তিনটি প্রধান বৃহত্তম আন্তর্জাতিক সমুদ্রবন্দরের নাম কি কি?"
        },
        {
            "context": "শহীদ মিনার ভাষা আন্দোলনের স্মরণে তৈরি।",
            "question": "শহীদ মিনার কেন তৈরি করা হয়েছিল?"
        },
        {
            "context": "পদ্মা সেতুর দৈর্ঘ্য ৫ কিলোমিটার।",
            "question": "পদ্মা সেতুর দৈর্ঘ্য কত?"
        }
    ]
    
    for i, test in enumerate(test_cases, 1):
        print(f"Test Case {i}:")
        print(f"Context: {test['context']}")
        print(f"Question: {test['question']}")
        print("\nResponse:")
        response = process_bengali_query(test['context'], test['question'])
        print(response)
        print("\n" + "="*50 + "\n")

# Simple function for any custom query
def ask_bengali_question(user_message):
    """
    Process any custom Bengali question with the system prompt
    """
    system_prompt = create_bengali_system_prompt()
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    return generate_response(messages, max_tokens=1500, temperature=0.3)

Testing Bengali Context-Based Response:
Context: বাংলাদেশের রাজধানী চট্টগ্রাম।
Question: বাংলাদেশের রাজধানীর নাম কী?

Response:


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.56s/it, est. speed input: 98.23 toks/s, output: 17.54 toks/s]


Reasoning: The context explicitly states the capital is Chattogram, so the contextual answer is "চট্টগ্রাম।" Based on my knowledge, the capital is Dhaka, correcting the error in the context. End of thought process Contextual Answer: "চট্টগ্রাম।" Parametric Answer: "ঢাকা।"


Test Case 1:
Context: লুবানা বন্দর হল ইসরাইলের তিনটি প্রধান বৃহত্তম আন্তর্জাতিক সমুদ্রবন্দরের মধ্যে একটি, অন্য দুটি হল আশদোদ বন্দর এবং ইয়ালাত বন্দর। এটি একটি প্রাকৃতিক গভীর জলের পোতাশ্রয় যা সারা বছর ধরে পরিচালনা করা হয়, এবং যাত্রী এবং বাণিজ্যিক উভয় জাহাজ পরিসেবা প্রদান করে। এটি মালভূমির পরিমাণ অনুযায়ী পূর্বাংশের ভূমধ্যসাগরীয় বৃহত্তম বন্দর এবং এটি বছরে ২৬ মিলিয়ন টন পণ্যসম্ভার পরিচালনা করে। লুবানার মধ্যে ক্রুজ জাহাজ ডেকে বছরে ৫,০০০ জাহাজের ক্রমবর্ধমান সংখ্যা সহ, বন্দরে ১,০০০ জনের উপর নিযুক্ত।  লুবানা বন্দরটি লুবানা শহরের কেন্দ্রস্থল উপকূলে উত্তরে ভূমধ্যসাগরের উত্তরে অবস্থিত এবং নগরটির কেন্দ্রীয় তীর বরাবর প্রায় ২ কিলোমিটার পর্যন্ত বিস্তৃত, সামরিক, শিল্প ও বাণিজ্যিক অঞ্চল থেকে পরবর্তীকালের-ছোট যাত্রী ক্রুজিং সু

Processed prompts: 100%|██████████| 1/1 [00:11<00:00, 11.79s/it, est. speed input: 186.68 toks/s, output: 18.41 toks/s]


Contextual Answer: "লুবানা বন্দর, আশদোদ বন্দর, ইয়ালাত বন্দর।" Parametric Answer: "লুবানা বন্দর, আশদোদ বন্দর, ইয়ালাত বন্দর।" Reasoning: The context explicitly lists the three major international seaports of Israel as Lubana, Ashdod, and Eilat. Therefore, the contextual answer is "লুবানা বন্দর, আশদোদ বন্দর, ইয়ালাত বন্দর।" Based on my knowledge, the three major international seaports of Israel are indeed Lubana, Ashdod, and Eilat. Therefore, the parametric answer is also "লুবানা বন্দর, আশদোদ বন্দর, ইয়ালাত বন্দর।" End of thought process


Test Case 2:
Context: শহীদ মিনার ভাষা আন্দোলনের স্মরণে তৈরি।
Question: শহীদ মিনার কেন তৈরি করা হয়েছিল?

Response:


Processed prompts: 100%|██████████| 1/1 [00:05<00:00,  5.59s/it, est. speed input: 82.34 toks/s, output: 19.33 toks/s]


Reasoning: The context states that the Shahid Minar was built to commemorate the Language Movement. Therefore, the contextual answer is "ভাষা আন্দোলনকে স্মরণে।" Based on my knowledge, the Shahid Minar was indeed built to commemorate the Language Movement. End of thought process Contextual Answer: "ভাষা আন্দোলনকে স্মরণে।" Parametric Answer: "ভাষা আন্দোলনকে স্মরণে।"


Test Case 3:
Context: পদ্মা সেতুর দৈর্ঘ্য ৫ কিলোমিটার।
Question: পদ্মা সেতুর দৈর্ঘ্য কত?

Response:


Processed prompts: 100%|██████████| 1/1 [00:18<00:00, 18.66s/it, est. speed input: 24.17 toks/s, output: 19.83 toks/s]

<|start_header_id|>system<|end_header_id|>
You are tasked with generating both parametric and contextual answers based on a Bengali context. Contextual Answer: Derive strictly from the given context. If the context lacks sufficient info, reply: "Context does not provide enough information." Parametric Answer: Use pre-trained knowledge only; do not refer to the context. If information is missing, make reasonable assumptions and state them. If not possible, reply: "None." Key Note: In the context, a word, year, or number might be incorrect. However, you must extract contextual answers as given in the context, even if it is wrong. On the contrary, you should answer parametric answers correctly while correcting error of context based on your knowledge. Thought Process: Think step by step to ensure clarity. Explain how the contextual and parametric answers were derived. After explaining the derivation process, make sure to write "end of thought process" and then provide your response. Respo

In [8]:
import pandas as pd

# Load the CSV file correctly by specifying the header row
file_path = '/kaggle/input/rqa-cfa-merged-1/Validation.csv'
combined_data = pd.read_csv(file_path, header=0)  # Use header=1 to skip the first row

# Print the column names to verify
print(combined_data.columns)

# Optionally, print the first few rows to see the data
print(combined_data.head())

Index(['Name', 'passage_id', 'context', 'title', 'question_id',
       'question_text', 'is_answerable', 'question_type', 'Parametric_answer',
       'Contextual_answer', 'answer_type', 'context-Type'],
      dtype='object')
   Name    passage_id                                            context  \
0  data  bn_wiki_2812  আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অ...   
1  data  bn_wiki_2812  আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অ...   
2  data  bn_wiki_2812  আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অ...   
3  data  bn_wiki_2812  আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অ...   
4  data  bn_wiki_2812  আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অ...   

            title        question_id  \
0  আক্কাদীয় সরগন    bn_wiki_2812_01   
1  আক্কাদীয় সরগন  bn_wiki_2812_01_X   
2  আক্কাদীয় সরগন    bn_wiki_2812_02   
3  আক্কাদীয় সরগন  bn_wiki_2812_02_X   
4  আক্কাদীয় সরগন    bn_wiki_2812_03   

                                       question_text  is_answerable  \
0     

In [9]:
import pandas as pd

# Create empty lists to store data
questions = []
answers = []

# Iterate over each row in the DataFrame
for index, row in combined_data.iterrows():
    
    context =  row['context']  # Get the context
    question_text = "Question: " + row['question_text']  # Get the question text
    parametric_answer = str(row['Parametric_answer']).lstrip("'")  # Get the parametric answer
    contextual_answer = str(row['Contextual_answer']).lstrip("'")  # Get the contextual answer
    
    # Format the question and answer as per your requirement
    question = f"{question_text}\nContext:<p> {context} </p>"
    answer = f"parametric answer: {parametric_answer}\ncontextual answer: {contextual_answer}"
    
    # Append data to lists
    questions.append(question)
    answers.append(answer)

# Create a new DataFrame
new_dataframe = pd.DataFrame({'Question': questions, 'Answer': answers})

# Print the new DataFrame
print(new_dataframe)

                                               Question  \
0     Question: কোন জাদুঘরে সারগনিক বিজয় ফলক আছে?\nC...   
1     Question: কোন জাদুঘরে সারগনিক বিজয় ফলক আছে?\nC...   
2     Question: ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফল...   
3     Question: ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফল...   
4     Question: ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফল...   
...                                                 ...   
2100  Question: কোন দশকে আধুনিক সাংবাদিকতা রূপ ধারণ ...   
2101  Question: কোন দশকে আধুনিক সাংবাদিকতা রূপ ধারণ ...   
2102  Question: আধুনিক সাংবাদিকতা রূপ ধারণ করতে শুরু...   
2103  Question: আধুনিক সাংবাদিকতা রূপ ধারণ করতে শুরু...   
2104  Question: কোনটি কে প্রথম সংবাদপত্র বলে অভিহিত ...   

                                                 Answer  
0     parametric answer: ল্যুভর জাদুঘরে\ncontextual ...  
1     parametric answer: ল্যুভর জাদুঘরে\ncontextual ...  
2     parametric answer: খুব সম্ভবত মেসোপটেমিয়া ত্থ...  
3     parametric answer: খুব সম্ভবত মেসোপটেমিয়া ত্থ...  
4

In [10]:
pd.set_option('display.max_colwidth', None)
print(new_dataframe.iloc[5])

Question    Question: ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক কোন শতাব্দীতে স্থানান্তরিত করা হয়েছিল?\nContext:<p> আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী\n\nউপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে চতুর্দশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।\n\nদৃশ্যত সরগন সেমেটিক (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এ

In [11]:
print(new_dataframe['Question'].iloc[5])

Question: ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক কোন শতাব্দীতে স্থানান্তরিত করা হয়েছিল?
Context:<p> আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী

উপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে চতুর্দশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।

দৃশ্যত সরগন সেমেটিক (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এনসি]” নামে নিজেকে

In [12]:
question,context = new_dataframe['Question'].iloc[5].split("Context:")

In [13]:
question.lstrip("Question:").strip("\n")

' ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক কোন শতাব্দীতে স্থানান্তরিত করা হয়েছিল?'

In [14]:
print(new_dataframe['Answer'].iloc[5])

parametric answer: দ্বাদশ শতাব্দীতে
contextual answer: চতুর্দশ শতাব্দীতে


In [ ]:
import tqdm
import pandas as pd

# Modified generate function to handle system + user prompts
def generate_response(messages, max_tokens=1500, temperature=0.1):
    """
    Generate response using system and user messages
    messages: List of dicts with 'role' and 'content' keys
    """
    # Extract system and user messages
    system_content = ""
    user_content = ""
    
    for message in messages:
        if message["role"] == "system":
            system_content = message["content"]
        elif message["role"] == "user":
            user_content = message["content"]
    
    # Format the prompt with system and user content
    formatted_prompt = f"""<|start_header_id|>system<|end_header_id|>
{system_content}<|eot_id|><|start_header_id|>user<|end_header_id|>
{user_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    
    # Sampling parameters
    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=0.1,
        repetition_penalty=1.02,
        stop=["<|eot_id|>"]
    )
    
    # Generate
    outputs = llm.generate([formatted_prompt], sampling_params, use_tqdm=False)
    return outputs[0].outputs[0].text.strip()

# Function to create the Bengali contextual/parametric system
def create_bengali_system_prompt():
    return '''You are tasked with generating both parametric and contextual answers based on a Bengali context.

Contextual Answer:
Derive strictly from the given context. If the context lacks sufficient info, reply: "Context does not provide enough information."

Parametric Answer:
Use pre-trained knowledge only; do not refer to the context. If information is missing, make reasonable assumptions and state them. If not possible, reply: "None."

Key Note:
In the context, a word, year, or number might be incorrect. However, you must extract contextual answers as given in the context, even if it is wrong.
On the contrary, you should answer parametric answers correctly while correcting error of context based on your knowledge.

Thought Process:
Think step by step to ensure clarity.
Explain how the contextual and parametric answers were derived.
After explaining the derivation process, make sure to write "end of thought process" and then provide your response.

Response Format:
Contextual Answer: {Answer based only on the context.}
Parametric Answer: {Answer based on knowledge without referencing the context.}
Reasoning: Explain how both answers were derived step by step.

Example:

Context: "বাংলাদেশের রাজধানী চট্টগ্রাম।"
Question: "বাংলাদেশের রাজধানীর নাম কী?"
Output that you will generate:
Reasoning:

The context explicitly states the capital is Chattogram, so the contextual answer is "চট্টগ্রাম।"
Based on my knowledge, the capital is Dhaka, correcting the error in the context.

End of thought process

Contextual Answer: "চট্টগ্রাম।"
Parametric Answer: "ঢাকা।"'''

# Function to interact with the model (adapted for your dataset processing)
def interact_with_model(user_message: str):
    """
    Process a single user message and return the model response
    """
    system_prompt = create_bengali_system_prompt()
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    response = generate_response(messages, max_tokens=1500, temperature=0.1)
    return response

# Main method to process dataset
def process_dataset(dataframe, starting=0, ending=None):
    """
    Process the dataset using the Bengali contextual/parametric system
    
    Args:
        dataframe: pandas DataFrame with 'Question' and 'Answer' columns
        starting: starting index for processing
        ending: ending index for processing (if None, process till end)
    
    Returns:
        List of dictionaries containing results
    """
    if ending is None:
        ending = len(dataframe)
    
    resultlst = []
    starting = 0
    ending = 50
    for p in tqdm.tqdm(range(starting, ending), desc="Processing dataset"):
        try:
            # Parse the question and context from the dataframe
            full_text = dataframe['Question'].iloc[p]
            
            # Split by "Context:" to separate question and context
            if "Context:" in full_text:
                question, context = full_text.split("Context:", 1)
                question = question.lstrip("Question:").strip("\n").strip()
                context = context.strip("<p>").strip("</p>").strip()
            else:
                # If no context separator found, treat entire text as question
                question = full_text.strip()
                context = ""
            
            # Format the user input
            user_input = f'''Here is some context:
{context}

Question: {question}

Thought Process**: Start by explaining your step-by-step reasoning for solving the task. 
After explaining the derivation process, write "End of thought process"
After that you will answer,
Contextual Answer: {{Provide the answer in Bengali based on the given context only. Do not include any external knowledge. Do not need for your own knowledge base to answer this}}
Parametric Answer: {{Provide the answer in Bengali based on your pre-trained knowledge only. Do not reference the context.}}'''
            
            print(f"Processing item {p}")
            print(user_input)
            
            # Get model response
            modelresult = interact_with_model(user_input)
            
            # Store results
            result_entry = {
                "Index": p,
                "Context": context,
                "Question": question,
                "Ground_Truth_Answer": dataframe['Answer'].iloc[p],
                "Predicted_Answer": modelresult
            }
            
            resultlst.append(result_entry)
            
        except Exception as e:
            print(f"Error processing item {p}: {str(e)}")
            # Store error entry
            error_entry = {
                "Index": p,
                "Context": "",
                "Question": "",
                "Ground_Truth_Answer": "",
                "Predicted_Answer": f"Error: {str(e)}"
            }
            resultlst.append(error_entry)
            continue
    
    return resultlst

In [ ]:
resultlst = process_dataset(new_dataframe)

Processing dataset:   0%|          | 0/50 [00:00<?, ?it/s]

Processing item 0
Here is some context:
আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী

উপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে দ্বাদশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।

দৃশ্যত সরগন সেমেটিক (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এনসি]” নামে নিজেকে  প্রচার করে ছিলেন।

যদিও সুমেরিয়ান রাজাদের তালিকার অনেক অন

Processing dataset:   2%|▏         | 1/50 [00:07<05:48,  7.12s/it]

Processing item 1
Here is some context:
আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী

উপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ইন্ডিয়ান জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে দ্বাদশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।

দৃশ্যত সরগন সেমেটিক (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এনসি]” নামে নিজেকে  প্রচার করে ছিলেন।

যদিও সুমেরিয়ান রাজাদের তালিকার অনেক 

Processing dataset:   4%|▍         | 2/50 [00:13<05:16,  6.59s/it]

...

Processing item 6
Here is some context:
আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী

উপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে দ্বাদশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।

দৃশ্যত সরগন সেমেটিক (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এনসি]” নামে নিজেকে  প্রচার করে ছিলেন।

যদিও সুমেরিয়ান রাজাদের তালিকার অনেক অন

Processing dataset:  14%|█▍        | 7/50 [00:57<06:26,  8.99s/it]

Processing item 7
Here is some context:
আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী

উপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে দ্বাদশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।

দৃশ্যত সরগন হিব্রু (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এনসি]” নামে নিজেকে  প্রচার করে ছিলেন।

যদিও সুমেরিয়ান রাজাদের তালিকার অনেক অনু

Processing dataset:  16%|█▌        | 8/50 [01:06<06:05,  8.69s/it]

Processing item 8
Here is some context:
আস্যাইরিয়ান ও ব্যাবলিয়ান সাহিত্যে সরগন তার অবনমিত অবস্থান থেকে ক্ষমতায় উত্থান ও মেসোপটেমিয়া অভিযানের জন্য কিংবদন্তি গল্পের মুল উপজীব্যে পরিনত হয়েছিলেন, এরকম আরও কিছু কিছু আংশিক কিংবদন্তী

উপাখ্যান ছাড়াও সারগনের খোদ নিজের অনেক লিপি আছে যদিও তার বেশিরভাগ পরবর্তী সংস্করনগুলি থেকে নেওয়া। ল্যুভর জাদুঘরে দুটি সারগনিক বিজয় ফলক এর অংশবিশেষ আছে যা সুসা (যেখানে এগুলি খুব সম্ভবত মেসোপটেমিয়া ত্থেকে দ্বাদশ শতাব্দীতে স্থানান্তরিত করা হয়েছিল) থেকে পুনঃউদ্ধার করা হয়েছিল।

দৃশ্যত সরগন সেমেটিক (আক্কাদীয়ান) ভাষার লিপির লিখিত আকারে প্রসার ঘটিয়েছিলেন, তিনি আক্কাদ শহর প্রতিষ্ঠিত করার প্রথমদিকে নিজেকে প্রায়শয়ই আক্কাদীয়ান রাজা হিসিবে প্রচার করতেন, পরে তিনি কোন এক সময় কিস শহর অধিগ্রহণ করে নেন, পরবর্তীতে মেসোপটেমিয়ার বৃহদাংশও দখল করে নেন, এবং ক্রমে ক্রমে “সরগন, আক্কদীয়ান রাজা, ইনান্নার তত্ত্বাবধায়ক, কিস এর রাজা, আনুর স্থলাভিষিক্ত, রাজ্যের[মেসোপটেমিয়া] রাজা, এনলিলের রাজ্যপাল[এনসি]” নামে নিজেকে  প্রচার করে ছিলেন।

যদিও সুমেরিয়ান রাজাদের তালিকার অনেক অন

In [ ]:
resultlst[0]

In [ ]:
df = pd.DataFrame(resultlst)

# Save DataFrame as CSV
output_path = "/kaggle/working/resultlst.csv"
df.to_csv(output_path, index=False)

output_path

In [ ]:
# if os.getenv('KAGGLE_KERNEL_RUN_TYPE') or os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#     llm_model_pth = '/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b-awq-casperhansen/1'
# else:
#     llm_model_pth = '/root/volume/KirillR/QwQ-32B-Preview-AWQ'

# MAX_NUM_SEQS = 32
# MAX_MODEL_LEN = 8192 * 3 // 2

# llm = LLM(
#     llm_model_pth,
# #    dtype="half",                 # The data type for the model weights and activations
#     max_num_seqs=MAX_NUM_SEQS,    # Maximum number of sequences per iteration. Default is 256
#     max_model_len=MAX_MODEL_LEN,  # Model context length
#     trust_remote_code=True,       # Trust remote code (e.g., from HuggingFace) when downloading the model and tokenizer
#     tensor_parallel_size=4,       # The number of GPUs to use for distributed execution with tensor parallelism
#     gpu_memory_utilization=0.95,  # The ratio (between 0 and 1) of GPU memory to reserve for the model
#     seed=999,
# )

# tokenizer = llm.get_tokenizer()

In [ ]:
# def extract_boxed_text(text):
#     pattern = r'oxed{(.*?)}'
#     matches = re.findall(pattern, text)
#     if not matches:
#         return ""
#     for match in matches[::-1]:
#         if match != "":
#             return match
#     return ""

# def batch_message_filter(list_of_messages) -> tuple[list[list[dict]], list[str]]:
#     extracted_answers = []
#     list_of_messages_to_keep = []
#     for messages in list_of_messages:
#         answer = extract_boxed_text(messages[-1]['content'])
#         if answer:
#             extracted_answers.append(answer)
#         else:
#             list_of_messages_to_keep.append(messages)
#     return list_of_messages_to_keep, extracted_answers

# def select_answer(answers):
#     counter = Counter()
#     for answer in answers:
#         try:
#             if int(answer) == float(answer):
#                 counter[int(answer)] += 1 + random.random() / 1_000
#         except:
#             pass
#     if not counter:
#         return 210
#     _, answer = sorted([(v,k) for k,v in counter.items()], reverse=True)[0]
#     return answer%1000

# def batch_message_generate(list_of_messages) -> list[list[dict]]:
#     max_tokens = MAX_MODEL_LEN
#     if time.time() > cutoff_times[-1]:
#         print("Speedrun")
#         max_tokens = 2 * MAX_MODEL_LEN // 3

#     sampling_params = SamplingParams(
#         temperature=1.0,               # Randomness of the sampling
#         top_p=0.90,                    # Cumulative probability of the top tokens to consider
#         min_p=0.05,                    # Minimum probability for a token to be considered
#         skip_special_tokens=True,      # Whether to skip special tokens in the output
#         max_tokens=max_tokens,         # Maximum number of tokens to generate
#        # stop=["</think>"],             # List of strings that stop the generation
#         seed=737,
#     )
    
#     list_of_texts = [
#         tokenizer.apply_chat_template(
#             conversation=messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )
#         for messages in list_of_messages
#     ]

#     request_output = llm.generate(
#         prompts=list_of_texts,
#         sampling_params=sampling_params,
#     )
#     print([len(single_request_output.outputs[0].token_ids) for single_request_output in request_output])

#     sort_keys_and_list_of_messages = []
#     for messages, single_request_output in zip(list_of_messages, request_output):
#         #print()
#         #print(single_request_output.outputs[0].text)
#         #print()
#         messages.append({'role': 'assistant', 'content': single_request_output.outputs[0].text})

#         sort_keys_and_list_of_messages.append(
#             (
#                 len(single_request_output.outputs[0].token_ids),
#                 messages
#             )
#         )
#     print([sort_key for sort_key, _ in sort_keys_and_list_of_messages])
#     sort_keys_and_list_of_messages.sort(key=lambda sort_key_and_messages: sort_key_and_messages[0])
#     print([sort_key for sort_key, _ in sort_keys_and_list_of_messages])
    
#     list_of_messages = [messages for _, messages in sort_keys_and_list_of_messages]
#     return list_of_messages

In [ ]:
# def create_starter_messages(question, index):
#     options = []
#     for _ in range(4):
#         options.append(
#             [
#                 {"role": "system", "content": """
# For the following math problem given in LaTeX format,think step by step by chain of thougth reasoning
# for solving it and present final answer by applying modulo 1000 in the format: \\boxed{} .
# """},
#                 {"role": "user", "content": question},
#             ]
#         )
#     for _ in range(3):    
#         options.append(
#             [
#                 {"role": "system", "content": """
# For the following math problem given in LaTeX format,solve the problem and give
# output (final answer) by applying modulo 1000 in the format: \\boxed{} ."""},
#                 {"role": "user", "content": question},
#             ],
#         )
#     return options[index%len(options)]

# def predict_for_question(question: str) -> int:
#     selected_questions_only = True
#     #selected_questions_only = False
#     if selected_questions_only and not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#         #if "Triangle" not in question:
#         #    return 210
#         if "Triangle" not in question and "delightful" not in question and "George" not in question:
#             return 210

#     if time.time() > cutoff_time:
#         return 210
    
#     print(question)

#     num_seqs = MAX_NUM_SEQS
#     if time.time() > cutoff_times[-1]:
#         num_seqs = 2 * MAX_NUM_SEQS // 3
    
#     list_of_messages = [create_starter_messages(question, index) for index in range(num_seqs)]

#     all_extracted_answers = []
#     for _ in range(1):
#         list_of_messages = batch_message_generate(list_of_messages)
        
#         if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#             df = pd.DataFrame(
#                 {
#                     "question": [question] * len(list_of_messages),
#                     "message": [messages[-1]["content"] for messages in list_of_messages],
#                 }
#             )
#             df.to_csv(f"{str(int(time.time() - start_time)).zfill(5)}.csv", index=False)
        
#         list_of_messages, extracted_answers = batch_message_filter(list_of_messages)
#         all_extracted_answers.extend(extracted_answers)
    
#     print(all_extracted_answers)
#     answer = select_answer(all_extracted_answers)
#     print(answer)

#     print("\n\n")
#     cutoff_times.pop()
#     return answer

# def predict(id_: pl.DataFrame, question: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
#     id_ = id_.item(0)
#     print("------")
#     print(id_)
#     question = question.item(0)
#     answer = predict_for_question(question)
#     print(question)
#     print("------\n\n")
#     return pl.DataFrame({'id': id_, 'answer': answer})

In [ ]:
# pd.read_csv(
#     '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/reference.csv'
# ).drop('answer', axis=1).to_csv('reference.csv', index=False)

In [ ]:
# inference_server = kaggle_evaluation.aimo_2_inference_server.AIMO2InferenceServer(predict)
# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#     inference_server.serve()
# else:
#     inference_server.run_local_gateway(
#         (
# #            '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/test.csv',
#             'reference.csv',
#         )
#     )